# 12 - Pipeline complet DEPP

Ce notebook synthétise toute la chaîne de production statistique :
1. génération de données ;
2. import ;
3. contrôle qualité ;
4. imputation ;
5. analyse descriptive ;
6. régression ;
7. psychométrie ;
8. restitution.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import statsmodels.api as sm
import numpy as np

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.simulate_depp import generate_student_dataset, inject_quality_issues
from src.data.loader import save_csv, data_path

# 1. Génération
raw = generate_student_dataset(n_students=15000, seed=42)
raw = inject_quality_issues(raw, missing_rate=0.03)
raw_path = data_path('data', 'raw', 'etude_lecture_6e.csv')
save_csv(raw, raw_path)

# 2. Nettoyage
df = raw.copy()
numeric_cols = ['score_lecture', 'ressources_num', 'age']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())
categorical_cols = ['sexe', 'pcs', 'academie', 'type_etab']
for col in categorical_cols:
    if col in df.columns:
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val.iloc[0])

clean_path = data_path('data', 'interim', 'etude_lecture_6e_clean.csv')
save_csv(df, clean_path)

# 3. Analyse descriptive
print('Moyenne globale :', round(df['score_lecture'].mean(), 2))
print('Moyenne par PCS :')
print(df.groupby('pcs')['score_lecture'].mean().sort_values(ascending=False).round(2))

# 4. Régression
df_model = df.copy()
df_model['sexe'] = df_model['sexe'].map({'F': 0, 'M': 1})
df_model['retard'] = df_model['retard'].astype(int)
dummy_pcs = pd.get_dummies(df_model['pcs'], prefix='pcs', drop_first=True).astype(float)
X = pd.concat([df_model[['sexe', 'retard']], dummy_pcs], axis=1)
y = df_model['score_lecture']
X = sm.add_constant(X)
# Ensure numeric and align indices
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)
y = pd.to_numeric(y, errors='coerce')
valid_idx = X.index[~(X.isnull().any(axis=1) | y.isnull())]
X = X.loc[valid_idx]
y = y.loc[valid_idx]
model = sm.OLS(y, X).fit()
print('\nRésumé du modèle :')
print(model.summary())

# 5. Psychométrie
latent = np.random.default_rng(42).normal(0, 1, len(df))
items = pd.DataFrame({
    'item_1': np.clip(latent + np.random.default_rng(1).normal(0, 0.8, len(df)), 0, 100),
    'item_2': np.clip(latent + np.random.default_rng(2).normal(0, 0.8, len(df)), 0, 100),
    'item_3': np.clip(latent + np.random.default_rng(3).normal(0, 0.9, len(df)), 0, 100),
    'item_4': np.clip(latent + np.random.default_rng(4).normal(0, 0.8, len(df)), 0, 100)
})
n_items = items.shape[1]
item_var = items.var(axis=0, ddof=1)
total_var = items.sum(axis=1).var(ddof=1)
alpha = (n_items / (n_items - 1)) * (1 - item_var.sum() / total_var)
print('\nAlpha de Cronbach :', round(alpha, 3))

print('\nPipeline complet exécuté avec succès.')


Moyenne globale : 66.96
Moyenne par PCS :
pcs
Cadre                         80.92
Professions_intermediaires    71.40
Agriculteur                   64.91
Employe                       64.57
Retraite                      61.82
Ouvrier                       58.23
Name: score_lecture, dtype: float64

Résumé du modèle :
                            OLS Regression Results                            
Dep. Variable:          score_lecture   R-squared:                       0.451
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     1764.
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        18:27:48   Log-Likelihood:                -58160.
No. Observations:               15015   AIC:                         1.163e+05
Df Residuals:                   15007   BIC:                         1.164e+05
Df Model:                           7             